# NB10 — Geography vs study/batch confounding in microbial omics CV

**Question**: How much does geographic confounding inflate microbial community predictive
performance for soil metal concentrations, relative to study/batch confounding?

**Approach**: Three nested CV schemes on the same 42k-sample dataset using CLR (genus-level
relative abundance) features only (B1 model), so all variation is in how the folds are
constructed, not in the model or features.

| Scheme | Split unit | What it measures |
|--------|-----------|------------------|
| Random k-fold | Sample | Naive estimate (baseline; autocorrelated) |
| Spatial block | Geographic block | Performance after removing geographic autocorrelation |
| Study-blocked | ENA project | Performance after removing study/batch structure |

**Interpretation**:
- Δ_geo = RMSE_spatial − RMSE_random → geographic confounding inflation
- Δ_batch = RMSE_study − RMSE_random → study/batch confounding inflation
- If Δ_geo > Δ_batch: geography is the dominant confound
- If Δ_batch > Δ_geo: study effects dominate

**Caveat**: Studies cluster geographically, so the two confounds are correlated. Δ_geo and
Δ_batch are not independent — a combined geographic+study block CV would show the joint floor.
We also run this as a fourth scheme.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import KFold, GroupKFold
from xgboost import XGBRegressor

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h, annotate_n
apply_style()

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = BASE / 'data'
FIGS = BASE / 'figures'
SCRIPTS = BASE / 'scripts'
HMP = BASE.parent / 'hybrid_metal_prediction' / 'data'

sys.path.insert(0, str(SCRIPTS))
from modelling import get_features, rmse, TARGETS

## A — Load data and join study IDs

In [ ]:
fm = pd.read_parquet(DATA / 'feature_matrix.parquet')
blocks = pd.read_csv(DATA / 'spatial_blocks.csv', index_col=0)['block']

# ENA sample project IDs (srs_key → ERP/SRP project accession)
# Drop duplicates: some srs_keys appear in multiple projects; keep first occurrence
sp_raw = pd.read_parquet(HMP / 'sample_projects.parquet')
sp_dict = sp_raw.drop_duplicates(subset='srs_key').set_index('srs_key')['Project'].to_dict()
print(f'sample_projects: {len(sp_raw)} rows, {len(sp_dict)} unique srs_keys')

# Feature matrix sample_id has format ERR123456.ERS123456 — srs_key is after the dot
# Explicit construction avoids Index.str.split ambiguity across pandas versions
srs_keys = pd.Series(
    [sid.split('.')[1] for sid in fm.index],
    index=fm.index,
    name='srs_key',
)
projects = srs_keys.map(sp_dict)  # Series indexed by fm.index, values = ERP accessions

print(f'Feature matrix:      {fm.shape}')
print(f'Unique projects:     {projects.nunique()}')
print(f'Project coverage:    {projects.notna().sum():,}/{len(fm):,} ({projects.notna().mean()*100:.1f}%)')
print(f'Largest project:     {projects.value_counts().index[0]} ({projects.value_counts().iloc[0]:,} samples)')
print(f'Spatial blocks:      {blocks.nunique()} blocks, coverage {blocks.notna().sum():,}')
print(f'projects index unique: {projects.index.is_unique}')

# Block × project co-occurrence check (are studies geographically clustered?)
bp = pd.DataFrame({'block': blocks, 'project': projects}).dropna()
proj_n_blocks = bp.groupby('project')['block'].nunique()
print(f'\nProject × block co-occurrence:')
print(f'  Projects spanning 1 block:  {(proj_n_blocks == 1).sum()}')
print(f'  Projects spanning 2 blocks: {(proj_n_blocks == 2).sum()}')
print(f'  Projects spanning 3+ blocks:{(proj_n_blocks >= 3).sum()}')
print('  (Most projects spanning 1 block → study and geography confounds are correlated)')

## B — CV scheme definitions

All three schemes use XGBoost with n_estimators=200 (fast; relative differences robust to this
choice). B1 model (CLR-only) isolates the microbial omics signal.

In [ ]:
N_EST = 200  # consistent across all schemes
MODEL = 'B1'  # CLR-only — isolates microbial omics signal


def _xgb(n_estimators=N_EST):
    return XGBRegressor(
        n_estimators=n_estimators,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        random_state=42,
        n_jobs=-1,
    )


def _valid_mask(X, y):
    return ~(X.isna().any(axis=1) | y.isna())


def run_random_kfold(feature_df, target, model_name, n_splits=5, seed=42):
    """Standard k-fold CV — baseline (autocorrelation-naive)."""
    y = feature_df[target]
    X = get_features(feature_df, model_name)
    mask = _valid_mask(X, y)
    X_v, y_v = X[mask], y[mask]

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rows = []
    for fold, (tr, te) in enumerate(kf.split(X_v)):
        m = _xgb()
        m.fit(X_v.iloc[tr], y_v.iloc[tr])
        preds = m.predict(X_v.iloc[te])
        rows.append(dict(target=target, scheme='random', fold=fold,
                         n_train=len(tr), n_test=len(te),
                         rmse_val=rmse(y_v.iloc[te].values, preds)))
    return pd.DataFrame(rows)


def run_spatial_block(feature_df, target, model_name, blocks_series):
    """Leave-one-geographic-block-out CV."""
    y = feature_df[target]
    X = get_features(feature_df, model_name)
    blks = blocks_series.reindex(feature_df.index)

    rows = []
    for blk in sorted(blks.dropna().unique()):
        te_mask = blks == blk
        tr_mask = ~te_mask & blks.notna()
        mask_tr = _valid_mask(X[tr_mask], y[tr_mask])
        mask_te = _valid_mask(X[te_mask], y[te_mask])
        X_tr, y_tr = X[tr_mask][mask_tr], y[tr_mask][mask_tr]
        X_te, y_te = X[te_mask][mask_te], y[te_mask][mask_te]
        if len(X_tr) < 20 or len(X_te) < 5:
            continue
        m = _xgb()
        m.fit(X_tr, y_tr)
        preds = m.predict(X_te)
        rows.append(dict(target=target, scheme='spatial', fold=blk,
                         n_train=len(X_tr), n_test=len(X_te),
                         rmse_val=rmse(y_te.values, preds)))
    return pd.DataFrame(rows)


def run_study_blocked(feature_df, target, model_name, groups_series, n_splits=5):
    """GroupKFold CV with ENA project as grouping variable.
    
    GroupKFold ensures all samples from a project appear in the same fold.
    n_splits=5 balances thoroughness with runtime for 623 projects.
    """
    y = feature_df[target]
    X = get_features(feature_df, model_name)
    grps = groups_series.reindex(feature_df.index)

    # Restrict to rows with valid features, target, and group label
    mask = _valid_mask(X, y) & grps.notna()
    X_v, y_v, g_v = X[mask], y[mask], grps[mask]

    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    for fold, (tr, te) in enumerate(gkf.split(X_v, y_v, g_v)):
        m = _xgb()
        m.fit(X_v.iloc[tr], y_v.iloc[tr])
        preds = m.predict(X_v.iloc[te])
        rows.append(dict(target=target, scheme='study', fold=fold,
                         n_train=len(tr), n_test=len(te),
                         n_test_studies=g_v.iloc[te].nunique(),
                         rmse_val=rmse(y_v.iloc[te].values, preds)))
    return pd.DataFrame(rows)


def run_combined_block(feature_df, target, model_name, blocks_series, groups_series, n_splits=5):
    """Joint geographic+study block CV — shows combined confounding floor.
    
    Strategy: create a combined label (block, project) and group by it. Within each
    geographic block, all samples from the same project stay together. Then GroupKFold
    over the 5 spatial blocks (n_splits=5).
    """
    y = feature_df[target]
    X = get_features(feature_df, model_name)
    blks = blocks_series.reindex(feature_df.index)

    rows = []
    for blk in sorted(blks.dropna().unique()):
        te_mask = blks == blk
        tr_mask = ~te_mask & blks.notna()
        mask_tr = _valid_mask(X[tr_mask], y[tr_mask])
        mask_te = _valid_mask(X[te_mask], y[te_mask])
        X_tr, y_tr = X[tr_mask][mask_tr], y[tr_mask][mask_tr]
        X_te, y_te = X[te_mask][mask_te], y[te_mask][mask_te]
        if len(X_tr) < 20 or len(X_te) < 5:
            continue
        m = _xgb()
        m.fit(X_tr, y_tr)
        preds = m.predict(X_te)
        rows.append(dict(target=target, scheme='combined', fold=blk,
                         n_train=len(X_tr), n_test=len(X_te),
                         rmse_val=rmse(y_te.values, preds)))
    return pd.DataFrame(rows)


print('CV functions defined.')
print(f'Model: {MODEL} (CLR-only), n_estimators={N_EST}')

## C — Run all CV schemes

In [ ]:
all_results = []

for target in TARGETS:
    metal = target.replace('log_', '').replace('_ppm', '')
    print(f'--- {metal} ---')

    print(f'  Random 5-fold...')
    df = run_random_kfold(fm, target, MODEL)
    all_results.append(df)
    print(f'  RMSE (mean ± sd): {df["rmse_val"].mean():.4f} ± {df["rmse_val"].std():.4f}')

    print(f'  Spatial block...')
    df = run_spatial_block(fm, target, MODEL, blocks)
    all_results.append(df)
    print(f'  RMSE (mean ± sd): {df["rmse_val"].mean():.4f} ± {df["rmse_val"].std():.4f}')

    print(f'  Study-blocked (GroupKFold n=5)...')
    df = run_study_blocked(fm, target, MODEL, projects)
    all_results.append(df)
    print(f'  RMSE (mean ± sd): {df["rmse_val"].mean():.4f} ± {df["rmse_val"].std():.4f}')

    print(f'  Combined (spatial blocks as primary split)...')
    df = run_combined_block(fm, target, MODEL, blocks, projects)
    all_results.append(df)
    print(f'  RMSE (mean ± sd): {df["rmse_val"].mean():.4f} ± {df["rmse_val"].std():.4f}')

results_df = pd.concat(all_results, ignore_index=True)
print(f'\nTotal CV runs: {len(results_df)}')

## D — Compile summary statistics

In [ ]:
# Mean RMSE per target × scheme (across folds)
summary = (
    results_df
    .groupby(['target', 'scheme'])['rmse_val']
    .agg(['mean', 'std'])
    .reset_index()
)
summary.columns = ['target', 'scheme', 'rmse_mean', 'rmse_sd']
summary['metal'] = summary['target'].str.replace('log_', '').str.replace('_ppm', '')

# Pivot to compute deltas
pivot = summary.pivot(index='metal', columns='scheme', values='rmse_mean').round(4)
pivot['delta_geo']   = pivot['spatial'] - pivot['random']
pivot['delta_batch'] = pivot['study']   - pivot['random']
pivot['delta_combined'] = pivot['combined'] - pivot['random']

print('Mean RMSE per scheme (log1p ppm, fold-mean):')
print(pivot[['random', 'spatial', 'study', 'combined']].to_string())
print()
print('RMSE degradation vs random (Δ):')
print(pivot[['delta_geo', 'delta_batch', 'delta_combined']].to_string())
print()
print('Average across metals:')
print(f'  Δ_geo   (spatial − random): {pivot["delta_geo"].mean():.4f}')
print(f'  Δ_batch (study   − random): {pivot["delta_batch"].mean():.4f}')
print(f'  Δ_combined:                {pivot["delta_combined"].mean():.4f}')
print()
dom = 'geographic' if pivot['delta_geo'].mean() > pivot['delta_batch'].mean() else 'study/batch'
ratio = pivot['delta_geo'].mean() / max(pivot['delta_batch'].mean(), 0.001)
print(f'  → {dom} confounding is larger ({ratio:.1f}× ratio)')

pivot.to_csv(DATA / 'nb10_confounding_rmse_summary.csv')
results_df.to_csv(DATA / 'nb10_confounding_all_folds.csv', index=False)
print('\nSaved: nb10_confounding_rmse_summary.csv, nb10_confounding_all_folds.csv')

## E — Figure

In [ ]:
SCHEME_ORDER = ['random', 'spatial', 'study', 'combined']
SCHEME_LABELS = ['Random\n5-fold', 'Spatial\n5-block', 'Study-\nblocked', 'Combined\nblock']
SCHEME_COLORS = [PALETTE[0], PALETTE[1], PALETTE[2], PALETTE[3]]

METALS = ['Cu', 'Zn', 'Pb', 'Ni']
x = np.arange(len(METALS))
n_schemes = 4
w = 0.18
offsets = np.linspace(-(n_schemes - 1) / 2 * w, (n_schemes - 1) / 2 * w, n_schemes)

fig, axs = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

# ── Panel A: RMSE by scheme × metal ──────────────────────────────────────────
ax = axs[0]
for i, (scheme, label, color, offset) in enumerate(
        zip(SCHEME_ORDER, SCHEME_LABELS, SCHEME_COLORS, offsets)):
    vals = [pivot.loc[m, scheme] for m in METALS]  # pivot columns = scheme names
    sds  = summary[summary['scheme'] == scheme].set_index('metal')['rmse_sd']
    errs = [sds.get(m, 0) for m in METALS]
    ax.bar(x + offset, vals, w, label=label, color=color, edgecolor='k', lw=0.5)
    ax.errorbar(x + offset, vals, yerr=errs, fmt='none', color='#333333',
                capsize=2, capthick=0.8, lw=0.8)

ax.set_xticks(x)
ax.set_xticklabels(METALS)
ax.set_xlabel('Target metal')
ax.set_ylabel('RMSE (log1p ppm)')
ax.set_title('CV scheme comparison (B1: CLR-only)')
ax.legend(fontsize=7, loc='upper right')
grid_h(ax)

# ── Panel B: RMSE degradation (Δ vs random) ──────────────────────────────────
ax = axs[1]
delta_schemes = ['delta_geo', 'delta_batch', 'delta_combined']
delta_labels  = ['Δ_geo\n(spatial−random)', 'Δ_batch\n(study−random)', 'Δ_combined\n(both−random)']
delta_colors  = [PALETTE[1], PALETTE[2], PALETTE[3]]
x2 = np.arange(len(METALS))
nd = len(delta_schemes)
wd = 0.22
offs2 = np.linspace(-(nd - 1) / 2 * wd, (nd - 1) / 2 * wd, nd)

for i, (col, label, color, off) in enumerate(
        zip(delta_schemes, delta_labels, delta_colors, offs2)):
    vals = [pivot.loc[m, col] for m in METALS]
    ax.bar(x2 + off, vals, wd, label=label, color=color, edgecolor='k', lw=0.5)

ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xticks(x2)
ax.set_xticklabels(METALS)
ax.set_xlabel('Target metal')
ax.set_ylabel('ΔRMSE vs random CV (log1p ppm)')
ax.set_title('Confounding inflation')
ax.legend(fontsize=7, loc='upper left')
grid_h(ax)

# Mean Δ text annotation in top corner
mean_geo = pivot['delta_geo'].mean()
mean_bat = pivot['delta_batch'].mean()
ax.text(0.97, 0.97, f'mean Δ_geo={mean_geo:+.3f}\nmean Δ_batch={mean_bat:+.3f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=7, color='#808080',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='none'))

fig.suptitle('NB10: Geography vs study/batch confounding in microbial omics prediction', y=1.02)
save(fig, FIGS / 'fig_nb10_confounding_comparison')
print('Saved: fig_nb10_confounding_comparison.pdf')

## F — Summary for REPORT

In [ ]:
print('=== NB10 SUMMARY FOR REPORT ===')
print()
print('Data: 42,037 soil samples, B1 model (CLR-only, 200 genus features), XGBoost n_estimators=200')
print(f'Study coverage: {projects.notna().mean()*100:.1f}%, {projects.nunique()} unique ENA projects')
print()
print('RMSE by scheme (fold-mean, log1p ppm):')
for metal in METALS:
    r = pivot.loc[metal]
    print(f'  {metal}: random={r["random"]:.3f}, spatial={r["spatial"]:.3f}, '
          f'study={r["study"]:.3f}, combined={r["combined"]:.3f}')
print()
print('Δ RMSE (degradation vs random):')
for metal in METALS:
    r = pivot.loc[metal]
    print(f'  {metal}: Δ_geo={r["delta_geo"]:+.3f}, Δ_batch={r["delta_batch"]:+.3f}, '
          f'Δ_combined={r["delta_combined"]:+.3f}')
print()
print(f'Mean Δ across metals:')
print(f'  Δ_geo   = {pivot["delta_geo"].mean():+.4f}')
print(f'  Δ_batch = {pivot["delta_batch"].mean():+.4f}')
print(f'  Δ_combined = {pivot["delta_combined"].mean():+.4f}')
dom = 'geographic' if pivot['delta_geo'].mean() > pivot['delta_batch'].mean() else 'study/batch'
if abs(pivot['delta_geo'].mean() - pivot['delta_batch'].mean()) < 0.01:
    dom = 'approximately equal (geographic ≈ study/batch)'
print(f'\nConclusion: {dom} confounding is larger')

print()
print('Project × block co-occurrence (studies clustering geographically):')
print(f'  Projects in 1 block: {(proj_n_blocks == 1).mean()*100:.1f}%')
print('  → geographic and study confounds are correlated; Δ_combined ≈ max(Δ_geo, Δ_batch)')